# 21 - Introduccion a ciencia de datos en R: visualizacion, densidad y CRISP-DM

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender ciencia de datos como proceso antes de usar algoritmos.
2. Aplicar CRISP-DM a ejemplos simples.
3. Interpretar histogramas y scatter plots en R.
4. Relacionar histogramas normalizados con densidad de probabilidad.
5. Crear regiones de interes para analizar datos.
6. Practicar con ejercicios teoricos y practicos.

> En esta clase todavia no entrenamos algoritmos. Primero observamos, describimos y formulamos buenas preguntas.

## Ruta de la sesion

1. Ciencia de datos antes de los algoritmos.
2. CRISP-DM como mapa de trabajo.
3. Dataset academico simulado.
4. Histogramas y densidad.
5. Scatter plots.
6. Regiones de interes.
7. CRISP-DM completo con ejemplo.
8. Ejercicios.

## 1) Ciencia de datos antes de los algoritmos

Ciencia de datos no empieza con modelos. Empieza con una pregunta clara.

Antes de usar algoritmos conviene responder:

1. Que decision queremos mejorar?
2. Que datos tenemos?
3. Que significa cada columna?
4. Hay errores, faltantes o sesgos?
5. Que graficas ayudan a entender el problema?
6. Que accion real podria tomarse con el analisis?

## 2) CRISP-DM en lenguaje practico

CRISP-DM organiza proyectos de datos en 6 fases:

1. **Entendimiento del problema**.
2. **Entendimiento de los datos**.
3. **Preparacion de datos**.
4. **Modelado o representacion**.
5. **Evaluacion**.
6. **Despliegue**.

En esta sesion, la fase de modelado no significa entrenar un algoritmo. Puede ser una grafica, una tabla o una region de interes bien justificada.

In [ ]:
crisp_dm <- data.frame(
  fase = c(
    "1. Entendimiento del problema",
    "2. Entendimiento de los datos",
    "3. Preparacion de datos",
    "4. Modelado o representacion",
    "5. Evaluacion",
    "6. Despliegue"
  ),
  pregunta_guia = c(
    "Que decision queremos mejorar?",
    "Que columnas tenemos y que significan?",
    "Que limpiamos, filtramos o transformamos?",
    "Que grafica, tabla o region resume el patron?",
    "El resultado ayuda a responder la pregunta?",
    "Como se comunica o usa el hallazgo?"
  ),
  producto_en_esta_sesion = c(
    "Pregunta clara y alcance limitado",
    "Revision de variables, rangos y posibles sesgos",
    "Tabla ordenada y variables derivadas sencillas",
    "Histogramas, scatter plots y regiones de interes",
    "Interpretacion escrita",
    "Reporte breve o recomendacion"
  )
)

crisp_dm

## 3) Dataset academico simulado

Cada fila representa un estudiante. Las columnas son:

- `horas_estudio`
- `horas_sueno`
- `asistencia_pct`
- `traslado_min`
- `calificacion`

Los datos son simulados para practicar sin usar informacion personal real.

In [ ]:
set.seed(21)
n <- 180

horas_estudio <- pmin(pmax(rnorm(n, mean = 4.2, sd = 1.7), 0.2), 9.5)
horas_sueno <- pmin(pmax(rnorm(n, mean = 6.7, sd = 1.0), 3.5), 9.5)
asistencia_pct <- pmin(pmax(rnorm(n, mean = 83, sd = 10), 45), 100)
traslado_min <- pmin(pmax(rgamma(n, shape = 3.0, scale = 12.0), 5), 120)
ruido <- rnorm(n, mean = 0, sd = 8)

calificacion <- 38 +
  5.6 * horas_estudio +
  0.24 * asistencia_pct +
  1.5 * (horas_sueno - 6.5) -
  0.04 * traslado_min +
  ruido

calificacion <- pmin(pmax(calificacion, 0), 100)

df <- data.frame(
  estudiante = 1:n,
  horas_estudio = round(horas_estudio, 1),
  horas_sueno = round(horas_sueno, 1),
  asistencia_pct = round(asistencia_pct, 1),
  traslado_min = as.integer(round(traslado_min, 0)),
  calificacion = round(calificacion, 1)
)

head(df)

In [ ]:
summary(df[, c(
  "horas_estudio",
  "horas_sueno",
  "asistencia_pct",
  "traslado_min",
  "calificacion"
)])

## 4) Histograma: ver una variable

Un histograma divide una variable numerica en intervalos (`bins`) y cuenta cuantas observaciones caen en cada intervalo.

Sirve para preguntar:

- Donde se concentran los valores?
- Que tan dispersos estan?
- Hay valores raros?
- La forma parece simetrica o cargada hacia un lado?

In [ ]:
hist(
  df$calificacion,
  breaks = seq(0, 100, by = 10),
  col = "steelblue",
  border = "white",
  main = "Histograma de calificaciones",
  xlab = "calificacion",
  ylab = "numero de estudiantes"
)

abline(v = mean(df$calificacion), col = "red", lwd = 2)
abline(v = median(df$calificacion), col = "darkgreen", lwd = 2, lty = 2)
legend(
  "topright",
  legend = c("media", "mediana"),
  col = c("red", "darkgreen"),
  lwd = 2,
  lty = c(1, 2)
)

## 5) Histograma normalizado y densidad

En una variable continua interpretamos probabilidades por rangos.

La idea central es:

$$P(a \leq X \leq b) \approx area bajo la densidad entre a y b$$

Un histograma normalizado aproxima una densidad. La probabilidad aproximada de una barra es:

$$altura \times ancho$$

In [ ]:
h <- hist(
  df$calificacion,
  breaks = seq(0, 100, by = 10),
  plot = FALSE
)

anchos <- diff(h$breaks)
densidad_aprox <- h$density
areas <- densidad_aprox * anchos

tabla_densidad <- data.frame(
  intervalo = paste(head(h$breaks, -1), "a", tail(h$breaks, -1)),
  densidad_aprox = round(densidad_aprox, 4),
  ancho = anchos,
  area_aprox = round(areas, 4)
)

sum(areas)
tabla_densidad

In [ ]:
hist(
  df$calificacion,
  breaks = seq(0, 100, by = 10),
  probability = TRUE,
  col = "skyblue",
  border = "white",
  main = "Densidad aproximada de calificacion",
  xlab = "calificacion",
  ylab = "densidad aproximada"
)

rect(70, 0, 90, max(h$density), col = rgb(1, 0.6, 0, 0.2), border = NA)
hist(
  df$calificacion,
  breaks = seq(0, 100, by = 10),
  probability = TRUE,
  col = rgb(0.3, 0.6, 0.9, 0.6),
  border = "white",
  add = TRUE
)

proporcion_70_90 <- mean(df$calificacion >= 70 & df$calificacion < 90)
legend(
  "topright",
  legend = paste("P(70 <= calificacion < 90) =", round(proporcion_70_90, 2)),
  bty = "n"
)

## 6) Scatter plot: relacion entre dos variables

Un scatter plot muestra una observacion como un punto.

Aqui:

- eje x: `horas_estudio`
- eje y: `calificacion`

Sirve para observar direccion, dispersion, casos raros y zonas donde se acumulan puntos.

In [ ]:
plot(
  df$horas_estudio,
  df$calificacion,
  pch = 19,
  col = rgb(0.1, 0.4, 0.7, 0.65),
  main = "Scatter plot: estudio y calificacion",
  xlab = "horas de estudio por semana",
  ylab = "calificacion"
)
grid()

In [ ]:
cor(
  df[, c(
    "horas_estudio",
    "horas_sueno",
    "asistencia_pct",
    "traslado_min",
    "calificacion"
  )]
)[, "calificacion"]

La correlacion resume relacion lineal, pero no reemplaza la grafica.

Una grafica puede mostrar dispersion, patrones curvos, grupos y casos raros que un solo numero no muestra.

Tambien recuerda: correlacion no implica causalidad.

## 7) Regiones de interes

Una region de interes es una zona del grafico con significado para la pregunta.

Ejemplos:

- calificacion menor a 60: posible riesgo academico.
- muchas horas de estudio pero calificacion baja: revisar habitos o contexto.
- alta asistencia y alta calificacion: patron de buen desempeno.

Estas regiones son transparentes y descriptivas; no son algoritmos de prediccion.

In [ ]:
df$region_interes <- "zona intermedia"
df$region_interes[df$calificacion < 60] <- "riesgo academico"
df$region_interes[df$horas_estudio >= 5 & df$calificacion < 70] <- "mucho estudio bajo resultado"
df$region_interes[df$horas_estudio < 3 & df$calificacion >= 75] <- "buen resultado con poco estudio"
df$region_interes[df$horas_estudio >= 5 & df$calificacion >= 80] <- "zona fuerte"

aggregate(
  cbind(calificacion, horas_estudio, asistencia_pct) ~ region_interes,
  data = df,
  FUN = function(x) round(mean(x), 1)
)

In [ ]:
colores <- c(
  "riesgo academico" = "red",
  "mucho estudio bajo resultado" = "orange",
  "buen resultado con poco estudio" = "purple",
  "zona fuerte" = "darkgreen",
  "zona intermedia" = "gray50"
)

plot(
  df$horas_estudio,
  df$calificacion,
  pch = 19,
  col = colores[df$region_interes],
  main = "Regiones de interes",
  xlab = "horas de estudio por semana",
  ylab = "calificacion"
)
abline(h = 60, lty = 2)
abline(h = 80, lty = 3)
abline(v = 5, lty = 2)
grid()
legend(
  "bottomright",
  legend = names(colores),
  col = colores,
  pch = 19,
  cex = 0.75
)

### Por que crear regiones

Crear regiones ayuda a:

1. Convertir una grafica en preguntas concretas.
2. Comparar subgrupos.
3. Detectar casos que requieren revision.
4. Comunicar hallazgos sin depender de codigo.
5. Preparar la intuicion para clases posteriores de algoritmos.

## 8) CRISP-DM completo con el ejemplo academico

Pregunta guia:

> Como podemos identificar patrones sencillos asociados con estudiantes que podrian necesitar apoyo academico?

La respuesta de esta sesion es exploratoria: graficas, resumenes y regiones de interes.

In [ ]:
crisp_ejemplo <- data.frame(
  fase = c(
    "Entendimiento del problema",
    "Entendimiento de los datos",
    "Preparacion de datos",
    "Modelado o representacion",
    "Evaluacion",
    "Despliegue"
  ),
  aplicacion_en_el_ejemplo = c(
    "Detectar patrones para orientar apoyo academico temprano.",
    "Revisar estudio, sueno, asistencia, traslado y calificacion.",
    "Crear una tabla limpia y una columna de region_interes.",
    "Usar histogramas, scatter plots y regiones transparentes.",
    "Ver si las regiones son comprensibles y utiles.",
    "Preparar un reporte breve para docentes o tutores."
  )
)

crisp_ejemplo

### Otro ejemplo simple: biblioteca universitaria

1. **Problema**: reducir tiempos de espera.
2. **Datos**: hora, dia, prestamos, usuarios en fila y tiempo de atencion.
3. **Preparacion**: quitar duplicados, agrupar por hora y revisar dias sin captura.
4. **Representacion**: histogramas de espera y scatter plot entre fila y espera.
5. **Evaluacion**: validar si las horas detectadas coinciden con la experiencia del personal.
6. **Despliegue**: recomendar horarios con mas personal durante semanas pico.

## 9) Ejercicios teoricos

### Ejercicio teorico 1: CRISP-DM

Elige asistencia a clases, comedor universitario, biblioteca o traslado al campus.

Para cada fase escribe una pregunta, una variable necesaria y una limitacion.

### Ejercicio teorico 2: histograma

Explica que es un bin, por que importa su ancho y que diferencia hay entre frecuencia y densidad.

### Ejercicio teorico 3: densidad

Explica por que el area bajo la densidad representa probabilidad en un rango.

### Ejercicio teorico 4: scatter plot

Describe direccion, dispersion, casos raros y una variable adicional que podria ayudar.

### Ejercicio teorico 5: regiones

Propone dos regiones de interes para un problema real y explica que decision apoyarian.

## 10) Ejercicios practicos

### Practica 1: cambiar bins

Cambia `ancho_bin` a 5, 10, 15 y 20. Describe que cambia.

In [ ]:
ancho_bin <- 10

hist(
  df$calificacion,
  breaks = seq(0, 100, by = ancho_bin),
  col = "steelblue",
  border = "white",
  main = paste("Histograma con bins de ancho", ancho_bin),
  xlab = "calificacion"
)

### Practica 2: probabilidad por rango

Cambia los limites e interpreta la proporcion.

In [ ]:
limite_inferior <- 70
limite_superior <- 80

mascara <- df$calificacion >= limite_inferior & df$calificacion < limite_superior
conteo <- sum(mascara)
proporcion <- mean(mascara)

conteo
proporcion

### Practica 3: otro scatter plot

Cambia `x` y `y` por otras columnas numericas.

In [ ]:
x <- "asistencia_pct"
y <- "calificacion"

plot(
  df[[x]],
  df[[y]],
  pch = 19,
  col = rgb(0.0, 0.6, 0.7, 0.7),
  main = paste("Scatter plot:", x, "vs", y),
  xlab = x,
  ylab = y
)
grid()

### Practica 4: definir una region propia

Crea una region booleana y cuenta cuantos estudiantes caen ahi.

In [ ]:
df$mi_region <- df$asistencia_pct < 75 & df$calificacion < 65

sum(df$mi_region)
mean(df$mi_region)
head(df[df$mi_region, c("estudiante", "asistencia_pct", "calificacion")])

### Practica 5: mini CRISP-DM propio

Llena la tabla para un caso propio sin usar algoritmos.

In [ ]:
mi_crisp_dm <- data.frame(
  fase = c(
    "Entendimiento del problema",
    "Entendimiento de los datos",
    "Preparacion de datos",
    "Modelado o representacion",
    "Evaluacion",
    "Despliegue"
  ),
  mi_respuesta = c("", "", "", "", "", "")
)

mi_crisp_dm

## 11) Cierre

Ideas clave:

1. Ciencia de datos convierte datos en decisiones mejor informadas.
2. CRISP-DM ordena el proceso.
3. Histogramas describen distribuciones.
4. La densidad se interpreta por areas en rangos.
5. Scatter plots muestran relaciones y dispersion.
6. Las regiones de interes hacen que una grafica sea accionable.
7. Antes de usar algoritmos, hay que saber describir e interpretar datos.